In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_data_dir():
    for root in [Path.cwd(), *Path.cwd().parents]:
        d = root / "data"
        if (d / "press_data_normal.csv").exists() and (d / "outlier_data.csv").exists():
            return d
    raise FileNotFoundError("data 폴더에서 press_data_normal.csv / outlier_data.csv를 찾지 못함")

DATA_DIR = find_data_dir()

FILES = {
    "normal": DATA_DIR / "press_data_normal.csv",
    "abnormal": DATA_DIR / "outlier_data.csv",
}

SIGNALS = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]
GAP_SEC = 0.15

print("DATA_DIR:", DATA_DIR)

SIGNALS = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]
GAP_SEC = 0.15

def load_and_clean(path, source):
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].copy()
    df["source_row"] = np.arange(len(df))
    n_raw = len(df)

    required = ["TimeStamp", *SIGNALS, "Equipment_state"]
    n_missing = int(df[required].isna().sum().sum())
    df = df.dropna(subset=required).copy()

    df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], errors="coerce")
    n_bad_time = int(df["TimeStamp"].isna().sum())
    if n_bad_time:
        raise ValueError(f"{source}: TimeStamp 파싱 실패 {n_bad_time}건")

    dup_cols = ["TimeStamp", *SIGNALS, "Equipment_state"]
    dup = df.duplicated(subset=dup_cols)
    n_dup = int(dup.sum())
    df = df.loc[~dup].reset_index(drop=True)

    df["elapsed_sec"] = (df["TimeStamp"] - df["TimeStamp"].iloc[0]).dt.total_seconds()
    df["dt_sec"] = df["TimeStamp"].diff().dt.total_seconds()

    gap = df["dt_sec"] > GAP_SEC
    non_inc = df["dt_sec"] <= 0
    seg_no = (gap | non_inc).cumsum()

    prefix = "N" if source == "normal" else "A"
    df["segment_id"] = prefix + seg_no.astype(str).str.zfill(4)
    df["pos_in_seg"] = df.groupby("segment_id", sort=False).cumcount()
    df["seg_len"] = df.groupby("segment_id", sort=False)["segment_id"].transform("size")
    df["source"] = source

    diag = {
        "rows_raw": n_raw,
        "missing": n_missing,
        "duplicates_removed": n_dup,
        "rows_clean": len(df),
        "time_start": df["TimeStamp"].iloc[0],
        "time_end": df["TimeStamp"].iloc[-1],
        "median_dt_sec": df["dt_sec"].dropna().median(),
        "large_gaps": int(gap.sum()),
        "non_increasing": int(non_inc.sum()),
        "segments": df["segment_id"].nunique(),
    }
    return df, diag

frames, diagnostics = [], {}

for source, path in FILES.items():
    df, diag = load_and_clean(path, source)
    frames.append(df)
    diagnostics[source] = diag

data = pd.concat(frames, ignore_index=True)

for source, d in diagnostics.items():
    print(f"\n[{source.upper()}]")
    for k, v in d.items():
        print(f"{k}: {v}")

display(data.head())

DATA_DIR: c:\Users\astra\Desktop\K_eng\data

[NORMAL]
rows_raw: 20000
missing: 0
duplicates_removed: 1
rows_clean: 19999
time_start: 2022-07-12 00:00:00.019000
time_end: 2022-07-12 01:16:55.828000
median_dt_sec: 0.1
large_gaps: 598
non_increasing: 0
segments: 599

[ABNORMAL]
rows_raw: 600
missing: 0
duplicates_removed: 0
rows_clean: 600
time_start: 2022-07-17 10:51:07.943000
time_end: 2022-07-17 10:53:53.540000
median_dt_sec: 0.1
large_gaps: 20
non_increasing: 0
segments: 21


,TimeStamp,AI0_Vibration,AI1_Vibration,AI2_Current,Equipment_state,source_row,elapsed_sec,dt_sec,segment_id,pos_in_seg,seg_len,source
0,2022-07-12 00:00:00.019,0.116967,-0.176403,192.338730,0,0,0.0,NaN,N0000,0,41,normal
1,2022-07-12 00:00:00.119,0.030080,-0.210895,217.004330,0,1,0.1,0.1,N0000,1,41,normal
2,2022-07-12 00:00:00.219,-0.045794,-0.029424,197.510620,0,2,0.2,0.1,N0000,2,41,normal
3,2022-07-12 00:00:00.319,0.047519,0.258642,156.607170,0,3,0.3,0.1,N0000,3,41,normal
4,2022-07-12 00:00:00.419,0.020351,0.173777,93.236934,0,4,0.4,0.1,N0000,4,41,normal


In [6]:
# 전처리 결과 저장
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

SAVE_PATH = PROCESSED_DIR / "preprocessed_data.csv"
data.to_csv(SAVE_PATH, index=False)

print("저장 완료:", SAVE_PATH)
print("rows:", len(data))

저장 완료: c:\Users\astra\Desktop\K_eng\data\processed\preprocessed_data.csv
rows: 20599
